In [13]:
import litellm
import json
from tavily import TavilyClient
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List, Optional
load_dotenv(override=True)

True

In [17]:
TAVILY_KEY = os.getenv("TAVILY_KEY")
if not TAVILY_KEY:
    raise ValueError("TAVILY_KEY not found in environment variables")

tavily = TavilyClient(api_key=TAVILY_KEY)


In [18]:
#defining tool for searching query through travily
def search_tavily(query: str):
    result = tavily.search(query)
    return json.dumps(result, indent=2)


In [20]:
print(search_tavily("What are the top 5 most popular programming languages in 2024?"))

{
  "query": "What are the top 5 most popular programming languages in 2024?",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://www.hackerrank.com/blog/most-popular-languages-2024/",
      "title": "The Most Popular Programming Languages of 2024 - HackerRank Blog",
      "content": "Python, a versatile and easy-to-read language, has surged in popularity for web development, data analysis, and artificial intelligence (AI) projects. C++ is a fast and powerful programming language widely used in system software, game development, and high-performance applications. While it comes in second for developer popularity, C++ is the third most in-demand programming language, an 8% drop from 2022. SQL climbed in rank to become the fourth most popular coding language among developers in 2023, though its total usage in programming language tests did decrease slightly during the same period. While JavaScript is primarily a front-end programmi

In [21]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_tavily",
            "description": "Search the web for recent information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query for web search"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

In [22]:
messages = [
    {
        "role": "user",
        "content": "What are the latest developments in OpenAI?"
    }
]

In [27]:
response = completion(
    model="gemini/gemini-3.1-flash-lite",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)


In [77]:
message = response.choices[0].message


In [62]:
#checking if tool was requested by model and executing it
if response.choices[0].finish_reason == "tool_calls":
    print(f'tool is called with tool name = {response.choices[0].message.tool_calls[0].function.name}')   

tool is called with tool name = search_tavily


In [80]:
if response.choices[0].finish_reason == "tool_calls":

    #extract tool call
    tool_call = response.choices[0].message.tool_calls[0]
    tool_name = tool_call.function.name
    tool_args = json.loads(tool_call.function.arguments)
    print(tool_args['query'])

    if tool_name == "search_tavily":
        result = search_tavily(**tool_args)
        print(f'Tool execution result: {result}')

        messages.append(message)
        # Add tool result message
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": result
            }
        )
    # -----------------------------
        # Second LLM call
        # -----------------------------

        final_response = completion(
            model="gemini/gemini-3.1-flash-lite",
            messages=messages
        )

        print("\nFINAL ANSWER:\n")
        print(final_response.choices[0].message.content)

else:
    print(response.choices[0].message.content)


    

latest news OpenAI developments October 2024
Tool execution result: {
  "query": "latest news OpenAI developments October 2024",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://www.linkedin.com/pulse/weeks-latest-generative-ai-updates-october-1-2024-symphonyai-omq0c",
      "title": "This week's latest generative AI updates - October 1, 2024 - LinkedIn",
      "content": "OpenAI Establishes Multi-Agent Research Team for Advanced AI Development: OpenAI is forming a research team focused on multi-agent systems",
      "score": 0.9994253,
      "raw_content": null
    },
    {
      "url": "https://medium.com/nlplanet/weekly-ai-news-october-7th-2024-1b96f913aa6b",
      "title": "Weekly AI News \u2014 October 7th 2024 | by Fabio Chiusano - Medium",
      "content": "OpenAI has launched a new \u201cCanvas\u201d interface for ChatGPT, enhancing user interaction with features like side-by-side text and code",
      "score": 0.999189

/Users/rakshit/auto-analyst/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp.../f', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(



FINAL ANSWER:

As of October 2024, OpenAI has made significant strides in its funding, product interface, and research initiatives. Here are the key developments:

### 1. Record-Breaking Funding Round
In early October, OpenAI successfully secured a massive **$6.6 billion in new funding**. This investment round has reportedly pushed the company's valuation to $157 billion. While there was significant interest from major tech players, companies like Apple reportedly withdrew from negotiations to participate in this specific round.

### 2. Introduction of "Canvas"
OpenAI launched a new interface feature for ChatGPT called **"Canvas."** This is designed to improve how users interact with the AI for writing and coding projects. Instead of a traditional chat-only window, Canvas opens a separate side-by-side workspace where users can directly edit text or code generated by ChatGPT, allowing for better collaboration and more precise refinements.

### 3. "ChatGPT Search"
OpenAI introduced **"C

In [67]:
tool_args

'{"query": "latest news OpenAI developments October 2024"}'

In [ ]:
# Step 1 — Define the tool
tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information on a topic",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to look up"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Perform a simple calculation given a mathematical expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The mathematical expression to calculate (e.g. '2 + 2 * (3/4)')"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "python_code_executor",
            "description": "Execute Python code and return the result",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "The Python code to execute (e.g. 'import math; math.sqrt(16)')"
                    }
                },
                "required": ["code"]
            }
        }
    }
]

In [117]:
messages = []


In [118]:
# Step 2 — Actually run the tool when Claude asks for it
def run_tool(tool_name : str, tool_args):
    if tool_name == "web_search":
        results = tavily.search(tool_args["query"], max_results=3)
        # Just return the text snippets, not the whole object
        return "\n\n".join([r["content"] for r in results["results"]])
    elif tool_name == "calculator":
        # WARNING: using eval can be dangerous in production code. This is just for demonstration purposes.
        try:
            return str(eval(tool_args["expression"]))
        except Exception as e:
            return f"Error evaluating expression: {e}"
    elif tool_name == "python_code_executor":
        # WARNING: using exec can be dangerous in production code. This is just for demonstration purposes.
        local_vars = {}
        try:
            exec(tool_args["code"], {}, local_vars)
            return str(local_vars.get("result", "No result variable set"))
        except Exception as e:
            return f"Error executing code: {e}"

# Step 3 — The agent loop
def run_agent(question):
    messages.append({"role": "user", "content": question})
    print(f"\nQuestion: {question}\n")

    while True:
        response = completion(
            model="gemini/gemini-3.1-flash-lite",   # swap to gpt-4o or anthropic/claude-sonnet-4-20250514
            messages=messages,
            tools=tools
        )
        # print(response.choices)

        choice = response.choices[0]

        # LLM wants to call a tool
        if choice.finish_reason == "tool_calls":
            tool_call = choice.message.tool_calls[0]
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(f"Tool called: {tool_name}")
            # print(f"Query: {tool_args['query']}\n")
            print(f"Args: {tool_args}")

            tool_result = run_tool(tool_name, tool_args)

            # Append assistant's tool call message first
            messages.append(choice.message)
            print(choice.message)

            # Then append the tool result
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            })

        # LLM is done — print final answer
        else:
            print(f"Answer:\n{choice.message.content}")
            break

        print("\n--- New LLM Response ---\n")

# run_agent("What is Vaxcyte VAX-31 and how does it compare to Merck CAPVAXIVE?")

In [119]:
run_agent("What is Merck's current stock price?")



Question: What is Merck's current stock price?

Tool called: web_search
Args: {'query': 'Merck stock price'}
Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(index=0, provider_specific_fields={'thought_signature': 'EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn'}, function=Function(arguments='{"query": "Merck stock price"}', name='web_search'), id='call_78c56c06bf3b4dbfab6c489ad7f6__thought__EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn', type='function')], function_call=None, images=[], thinking_blocks=[], provider_specific_fields={'thought_signatures': ['EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn']})

--- New LLM Response ---



/Users/rakshit/auto-analyst/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp...mn', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Answer:
Merck & Co., Inc. (MRK) is currently trading at **$111.35**.

*Note: Stock market data fluctuates throughout the trading day. For the most up-to-the-minute price and detailed market performance, please check a financial news website or your brokerage platform.*


In [120]:
print(messages)

[{'role': 'user', 'content': "What is Merck's current stock price?"}, Message(content=None, role='assistant', tool_calls=[{'index': 0, 'provider_specific_fields': {'thought_signature': 'EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn'}, 'function': {'arguments': '{"query": "Merck stock price"}', 'name': 'web_search'}, 'id': 'call_78c56c06bf3b4dbfab6c489ad7f6__thought__EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn', 'type': 'function'}], function_call=None, images=[], thinking_blocks=[], provider_specific_fields={'thought_signatures': ['EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn']}), {'role': 'tool', 'tool_call_id': 'call_78c56c06bf3b4dbfab6c489ad7f6__thought__EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn', 'content': "Merck & Co., Inc.'s stock was trading at $105.20 at the beginning of the year. Since then, MRK stock has increased by 5.8% and is now trading at $111.2550.\n\n# Me

In [121]:
run_agent("What about Pfizer?")  # does it remember context?


Question: What about Pfizer?



/Users/rakshit/auto-analyst/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp...mn', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Tool called: web_search
Args: {'query': 'Pfizer stock price'}
Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(index=0, provider_specific_fields={'thought_signature': 'EjQKMgEMOdbH5FUZOVFM/u+6Oee4HruxtOJPgVldRYsNDUOfo/a1O3yiirZ1FTXq4plNzvmk'}, function=Function(arguments='{"query": "Pfizer stock price"}', name='web_search'), id='call_d2c0a774b3e04a898dc00a395e81__thought__EjQKMgEMOdbH5FUZOVFM/u+6Oee4HruxtOJPgVldRYsNDUOfo/a1O3yiirZ1FTXq4plNzvmk', type='function')], function_call=None, images=[], thinking_blocks=[], provider_specific_fields={'thought_signatures': ['EjQKMgEMOdbH5FUZOVFM/u+6Oee4HruxtOJPgVldRYsNDUOfo/a1O3yiirZ1FTXq4plNzvmk']})

--- New LLM Response ---



/Users/rakshit/auto-analyst/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp...mk', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Answer:
Pfizer (PFE) is currently trading at **$26.46**.

*Note: Stock prices change constantly throughout the trading day. The figures provided are based on recent market data.*


In [122]:
run_agent("Which one is higher?")  # comparing?


Question: Which one is higher?

Answer:
Based on the information provided:

*   **Merck (MRK)** is trading at **$111.35**.
*   **Pfizer (PFE)** is trading at **$26.46** (with recent data showing ranges between $25.68 and $26.46).

**Merck's stock price is significantly higher** than Pfizer's.


## planner agent added

In [126]:


# load_dotenv()
# tavily = TavilyClient()
model="gemini/gemini-3.1-flash-lite"
# ─── Pydantic Models ───────────────────────────────────────────────
class SubTask(BaseModel):
    task: str
    tool_name: str
    tool_args: dict

class TaskPlan(BaseModel):
    subtasks: List[SubTask]

# ─── Tools ────────────────────────────────────────────────────────
tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information on a topic",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression to evaluate"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "code_exec",
            "description": "Execute arbitrary Python code and return the output",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {"type": "string", "description": "Python code to execute"}
                },
                "required": ["code"]
            }
        }
    }
]


In [129]:
# ─── Planner ──────────────────────────────────────────────────────
def run_planner(question: str) -> TaskPlan:
    system_prompt = """You are a planner. Your job is to break a user question into subtasks.
Each subtask must use exactly one of these tools: web_search, calculator, code_exec.

Return ONLY a valid JSON object in this exact format — no prose, no markdown, no backticks:
{
  "subtasks": [
    {
      "task": "plain English description of what this step does",
      "tool_name": "web_search | calculator | code_exec",
      "tool_args": { "query": "..." }
    }
  ]
}

tool_args must match the tool:
- web_search  → { "query": "..." }
- calculator  → { "expression": "..." }
- code_exec   → { "code": "..." }
"""

    response = litellm.completion(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )

    raw = response.choices[0].message.content
    parsed = json.loads(raw)
    return TaskPlan(**parsed)  # validates against Pydantic model

# ─── Executor ─────────────────────────────────────────────────────
messages = []

def run_agent(question: str):
    messages.append({"role": "user", "content": question})
    print(f"\nQuestion: {question}\n")

    while True:
        response = litellm.completion(
            model=model,
            messages=messages,
            tools=tools
        )

        choice = response.choices[0]

        if choice.finish_reason == "tool_calls":
            tool_call = choice.message.tool_calls[0]
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(f"Tool called: {tool_name}")
            print(f"Args: {tool_args}\n")

            tool_result = run_tool(tool_name, tool_args)

            messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": choice.message.tool_calls
                            })
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            })

        else:
            print(f"Answer:\n{choice.message.content}")
            messages.append({"role": "assistant", "content": choice.message.content})
            break

# ─── Planner + Executor Pipeline ──────────────────────────────────
def run_pipeline(question: str):
    print("\n========== PLANNER ==========")
    plan = run_planner(question)

    for i, subtask in enumerate(plan.subtasks):
        print(f"\nSubtask {i+1}: {subtask.task}")
        print(f"Tool: {subtask.tool_name} | Args: {subtask.tool_args}")

    print("\n========== EXECUTOR ==========")
    # Feed the full plan as context to the executor
    plan_summary = "\n".join(
        [f"Step {i+1}: {s.task} using {s.tool_name}" for i, s in enumerate(plan.subtasks)]
    )
    enriched_question = f"{question}\n\nHere is your execution plan:\n{plan_summary}"
    run_agent(enriched_question)



In [130]:
# ─── Run ──────────────────────────────────────────────────────────
run_pipeline("What is Merck's current stock price, multiply it by 47 shares, then subtract 12.5% tax on gains assuming I bought at $85?")


========== PLANNER ==========

Subtask 1: Find the current stock price of Merck (MRK)
Tool: web_search | Args: {'query': 'current stock price of Merck MRK'}

Subtask 2: Calculate the total value of 47 shares, the total gain from the purchase price of $85, the tax on that gain, and the final net amount
Tool: code_exec | Args: {'code': "current_price = 127.35 # Placeholder value, will be updated by runtime data\nshares = 47\nbuy_price = 85\ntotal_value = current_price * shares\ntotal_gain = (current_price - buy_price) * shares\ntax = total_gain * 0.125\nnet_profit = total_gain - tax\nprint(f'{total_value=}, {net_profit=}')"}

========== EXECUTOR ==========

Question: What is Merck's current stock price, multiply it by 47 shares, then subtract 12.5% tax on gains assuming I bought at $85?

Here is your execution plan:
Step 1: Find the current stock price of Merck (MRK) using web_search
Step 2: Calculate the total value of 47 shares, the total gain from the purchase price of $85, the tax o

In [140]:
from generate_mmm_output import sample_roi,VENDOR_ROI,build_channel,SPEND_RANGES

In [138]:
sample_roi('doximity', 'oncology')

3.07

In [146]:
np.random.randint(*SPEND_RANGES["doximity"])

1154339

In [5]:
# imports

import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm


/Users/rakshit/auto-analyst/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import json
import os
from chromadb.utils import embedding_functions

In [28]:
# Point to your generated data
DATA_FOLDER = "mmm_dummy_data" #setting the folder name where data is stored

# Load one file first — understand it before looping all 50
sample_file = os.listdir(DATA_FOLDER)[0]
with open(f"{DATA_FOLDER}/{sample_file}") as f:
    data = json.load(f)

print(json.dumps(data,indent=2))

{
  "model_id": "pharma_vaccines_vaxneuvance_2024_16",
  "metadata": {
    "industry": "pharma",
    "sub_vertical": "vaccines",
    "brand": "vaxneuvance",
    "modelling_period": {
      "start": "2022-01",
      "end": "2024-12"
    },
    "reporting_period": {
      "start": "2024-01",
      "end": "2024-12"
    }
  },
  "financials": {
    "nrv": 56220997,
    "pgm": 34704660
  },
  "model_summary": {
    "total_revenue": 56220997,
    "base_contribution_pct": 0.56,
    "incremental_contribution_pct": 0.44,
    "r_squared": 0.9,
    "mape": 0.06
  },
  "channels": [
    {
      "name": "salesforce_calls",
      "category": "hcp",
      "tactics": [
        "primary_care_calls",
        "specialty_calls"
      ],
      "spend": 7569971,
      "calls_delivered": 69296,
      "reach": 17717,
      "frequency": 3.2,
      "transformation": {
        "adstock_decay_rate": 0.51
      },
      "coefficient": 0.0077,
      "roi": 1.86,
      "revenue_contribution": 14080146
    },
    {
 

In [7]:
import chromadb

# Initialize with persistent storage (do this EVERY session)
client = chromadb.PersistentClient(path="./mmm_vectorstore")  # points to your stored data

# Get existing collection (don't use get_or_create if you want to fail loudly)
collection = client.get_collection(name="mmm_outputs")

# Verify it loaded correctly
print(collection.count())  # should show your stored doc count

406


In [17]:
meta = data['metadata']
summary = data['model_summary']
fin = data['financials']

summary_text = (
    f"{meta['brand'].title()} is a {meta['sub_vertical']} brand. "
    f"The MMM model covers {meta['modelling_period']['start']} to {meta['modelling_period']['end']}. "
    f"Total revenue is ${fin['nrv']:,}. "
    f"Base contribution is {summary['base_contribution_pct']*100:.0f}% "
    f"and incremental contribution is {summary['incremental_contribution_pct']*100:.0f}%. "
    f"Model R-squared is {summary['r_squared']} with MAPE of {summary['mape']}."
)

In [22]:
# Pick the first channel from the sample file
channel = data["channels"][0]
meta = data["metadata"]

channel_text = (
    f"{channel['name'].title()} is a {channel['category'].upper()} channel "
    f"for {meta['brand'].title()} ({meta['sub_vertical']}). "
    f"Spend was ${channel['spend']:,} with an ROI of {channel['roi']}x. "
    f"Revenue contribution was ${channel['revenue_contribution']:,}. "
    f"Reach was {channel['reach']:,} with frequency of {channel['frequency']}. "
    f"Adstock decay rate is {channel['transformation']['adstock_decay_rate']}. "
    f"Tactics used: {', '.join(channel['tactics'])}."
)

# Add calls_delivered only if it exists
if channel.get("calls_delivered"):
    channel_text += f" Calls delivered: {channel['calls_delivered']:,}."

print(channel_text)

Salesforce_Calls is a HCP channel for Vaxneuvance (vaccines). Spend was $7,569,971 with an ROI of 1.86x. Revenue contribution was $14,080,146. Reach was 17,717 with frequency of 3.2. Adstock decay rate is 0.51. Tactics used: primary_care_calls, specialty_calls. Calls delivered: 69,296.


In [70]:
def chunk_mmm_file(filepath):
    with open(filepath) as f:
        data = json.load(f)

    meta = data['metadata']
    summary = data['model_summary']
    fin = data['financials']
    brand    = meta["brand"]
    vertical = meta["sub_vertical"]
    year     = meta["modelling_period"]["end"].split("-")[0]
    model_id = data["model_id"]

    chunks = []

    # Summary chunk
    summary_text = (
        f"{brand.title()} is a {vertical} brand. "
        f"Total revenue is ${fin['nrv']:,}. "
        f"The MMM model covers {meta['modelling_period']['start']} to {meta['modelling_period']['end']}. "
        f"Base contribution is {summary['base_contribution_pct']*100:.0f}% "
        f"and incremental contribution is {summary['incremental_contribution_pct']*100:.0f}%. "
        f"Model R-squared is {summary['r_squared']} with MAPE of {summary['mape']}."
    )
    chunks.append({
        "text": summary_text,
        "metadata": {
            "type": "summary",
            "brand": brand,
            "sub_vertical": vertical,
            "year": year,
            "model_id": model_id,
            "channel": "all",
            "category": "all"
        },
        "id": f"{model_id}_summary"
    })

    # Channel chunks
    for ch in data["channels"]:
        channel_text = (
            f"{ch['name'].title()} is a {ch['category'].upper()} channel "
            f"for {brand.title()} ({vertical}). "
            f"Revenue contribution was ${ch['revenue_contribution']:,}. "
            f"Spend was ${ch['spend']:,} with an ROI of {ch['roi']}x. "
            
            f"Reach was {ch['reach']:,} with frequency of {ch['frequency']}. "
            f"Adstock decay rate is {ch['transformation']['adstock_decay_rate']}. "
            f"Tactics used: {', '.join(ch['tactics'])}."
        )
        if ch.get("calls_delivered"):
            channel_text += f" Calls delivered: {ch['calls_delivered']:,}."
        
        chunks.append({
            "text": channel_text,
            "metadata": {
                "type": "channel",
                "brand": brand,
                "sub_vertical": vertical,
                "year": year,
                "model_id": model_id,
                "channel": ch["name"],
                "category": ch["category"]
            },
            "id": f"{model_id}_{ch['name']}"
        })
    
    return chunks



In [71]:
# Test it on one file
chunks = chunk_mmm_file(f"{DATA_FOLDER}/{sample_file}")
print(f"Total chunks from one file: {len(chunks)}")
for c in chunks:
    print(f"\n[{c['metadata']['type'].upper()}] {c['id']}")
    print(c['text'])

Total chunks from one file: 9

[SUMMARY] pharma_vaccines_vaxneuvance_2024_16_summary
Vaxneuvance is a vaccines brand. Total revenue is $56,220,997. The MMM model covers 2022-01 to 2024-12. Base contribution is 56% and incremental contribution is 44%. Model R-squared is 0.9 with MAPE of 0.06.

[CHANNEL] pharma_vaccines_vaxneuvance_2024_16_salesforce_calls
Salesforce_Calls is a HCP channel for Vaxneuvance (vaccines). Revenue contribution was $14,080,146. Spend was $7,569,971 with an ROI of 1.86x. Reach was 17,717 with frequency of 3.2. Adstock decay rate is 0.51. Tactics used: primary_care_calls, specialty_calls. Calls delivered: 69,296.

[CHANNEL] pharma_vaccines_vaxneuvance_2024_16_sfmc
Sfmc is a HCP channel for Vaxneuvance (vaccines). Revenue contribution was $129,686. Spend was $59,489 with an ROI of 2.18x. Reach was 25,503 with frequency of 3.7. Adstock decay rate is 0.48. Tactics used: hq_emails.

[CHANNEL] pharma_vaccines_vaxneuvance_2024_16_pulsepoint
Pulsepoint is a HCP channel 

In [72]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

client = chromadb.PersistentClient(path="./mmm_vectorstore")

collection = client.get_or_create_collection(
    name="mmm_outputs",
    embedding_function=embedding_fn
)

print(f"Collection created: {collection.name}")
print(f"Documents in collection: {collection.count()}")

Collection created: mmm_outputs
Documents in collection: 406


In [73]:
# Don't ingest all 50 yet — do one file and query it
chunks = chunk_mmm_file(f"{DATA_FOLDER}/{sample_file}")

collection.add(
    documents=[c["text"] for c in chunks],
    metadatas=[c["metadata"] for c in chunks],
    ids=[c["id"] for c in chunks]
)

print(f"Ingested {len(chunks)} chunks")
print(f"Total in collection now: {collection.count()}")

Ingested 9 chunks
Total in collection now: 406


In [75]:
# This is the most important cell — does retrieval actually work?
results = collection.query(
    query_texts=["what is the revenue of Proquad 2024?"],
    n_results=3
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Result {i+1} ---")
    print(doc)
    print(f"Metadata: {results['metadatas'][0][i]}")


--- Result 1 ---
Display is a CONSUMER channel for Proquad (vaccines). Spend was $245,750 with an ROI of 1.23x. Revenue contribution was $302,272. Reach was 6,500,976 with frequency of 3.8. Adstock decay rate is 0.24. Tactics used: banners.
Metadata: {'brand': 'proquad', 'model_id': 'pharma_vaccines_proquad_2024_26', 'sub_vertical': 'vaccines', 'type': 'channel', 'category': 'consumer', 'year': '2024', 'channel': 'display'}

--- Result 2 ---
Display is a CONSUMER channel for Proquad (vaccines). Spend was $255,608 with an ROI of 1.3x. Revenue contribution was $332,290. Reach was 6,665,618 with frequency of 3.4. Adstock decay rate is 0.28. Tactics used: banners.
Metadata: {'category': 'consumer', 'brand': 'proquad', 'type': 'channel', 'channel': 'display', 'model_id': 'pharma_vaccines_proquad_2024_18', 'sub_vertical': 'vaccines', 'year': '2024'}

--- Result 3 ---
Proquad is a vaccines brand. The MMM model covers 2022-01 to 2024-12. Total revenue is $110,542,173. Base contribution is 59%

In [76]:
results = collection.query(query_texts=["which channels have highest revenue contribution?"],n_results=3)

In [77]:
results['documents']

[['Streaming_Tv is a CONSUMER channel for Vaxelis (vaccines). Spend was $3,973,124 with an ROI of 2.22x. Revenue contribution was $8,820,335. Reach was 5,775,855 with frequency of 1.6. Adstock decay rate is 0.37. Tactics used: ctv, ott.',
  'Streaming_Tv is a CONSUMER channel for Vaqta (vaccines). Spend was $1,147,511 with an ROI of 3.27x. Revenue contribution was $3,752,360. Reach was 2,276,570 with frequency of 1.6. Adstock decay rate is 0.35. Tactics used: ctv, ott.',
  'Streaming_Tv is a CONSUMER channel for Qliftara (oncology). Spend was $1,235,226 with an ROI of 1.97x. Revenue contribution was $2,433,395. Reach was 2,338,291 with frequency of 2.8. Adstock decay rate is 0.41. Tactics used: ctv, ott.']]

In [3]:
def ingest_all(folder_path):
    all_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]
    
    for i, filename in enumerate(all_files):
        filepath = os.path.join(folder_path, filename)
        chunks = chunk_mmm_file(filepath)
        
        collection.add(
            documents=[c["text"] for c in chunks],
            metadatas=[c["metadata"] for c in chunks],
            ids=[c["id"] for c in chunks]
        )
        print(f"[{i+1}/{len(all_files)}] Ingested {filename} — {len(chunks)} chunks")
    
    print(f"\nDone. Total chunks in collection: {collection.count()}")

ingest_all(DATA_FOLDER)

NameError: name 'DATA_FOLDER' is not defined

In [79]:
result = collection.query(
    query_texts=["total revenue and model performance"],
    n_results=3,
    where={"$and": [{"brand": {"$eq": "proquad"}}, {"year": {"$eq": "2024"}}]}
)

In [80]:
result['documents']

[['Display is a CONSUMER channel for Proquad (vaccines). Spend was $255,608 with an ROI of 1.3x. Revenue contribution was $332,290. Reach was 6,665,618 with frequency of 3.4. Adstock decay rate is 0.28. Tactics used: banners.',
  'Display is a CONSUMER channel for Proquad (vaccines). Spend was $245,750 with an ROI of 1.23x. Revenue contribution was $302,272. Reach was 6,500,976 with frequency of 3.8. Adstock decay rate is 0.24. Tactics used: banners.',
  'Proquad is a vaccines brand. The MMM model covers 2022-01 to 2024-12. Total revenue is $110,542,173. Base contribution is 59% and incremental contribution is 41%. Model R-squared is 0.95 with MAPE of 0.09.']]

In [ ]:
CATEGORIES = []
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [98]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser" 
import numpy as np

# ── Build labels and colors ───────────────────────────────────────
# Color by sub_vertical
verticals  = [m["sub_vertical"] for m in metadatas]
brands     = [m["brand"] for m in metadatas]
types      = [m["type"] for m in metadatas]
channels   = [m["channel"] for m in metadatas]
years      = [m["year"] for m in metadatas]

vertical_palette = {
    "oncology": "#e63946",
    "vaccines":  "#2a9d8f",
    "pharma":    "#e9c46a",
}
colors = [vertical_palette.get(v, "#aaa") for v in verticals]

# ── Marker shape by chunk type ────────────────────────────────────
# summary → star, channel → circle
symbols = ["star" if t == "summary" else "circle" for t in types]

# ── Hover text ────────────────────────────────────────────────────
hover_texts = [
    f"<b>{brands[i].title()} — {verticals[i]}</b><br>"
    f"Type: {types[i]}<br>"
    f"Channel: {channels[i]}<br>"
    f"Year: {years[i]}<br>"
    f"<br><i>{documents[i][:120]}...</i>"
    for i in range(len(documents))
]

# ── One trace per vertical (for legend) ──────────────────────────
fig = go.Figure()

for vertical, color in vertical_palette.items():
    mask = [i for i, v in enumerate(verticals) if v == vertical]

    # summary points — stars
    summary_idx = [i for i in mask if types[i] == "summary"]
    fig.add_trace(go.Scatter(
        x=reduced[summary_idx, 0],
        y=reduced[summary_idx, 1],
        mode="markers",
        name=f"{vertical} — summary",
        marker=dict(
            symbol="star",
            size=12,
            color=color,
            opacity=0.9,
            line=dict(width=1, color="white")
        ),
        text=[hover_texts[i] for i in summary_idx],
        hoverinfo="text"
    ))

    # channel points — circles
    channel_idx = [i for i in mask if types[i] == "channel"]
    fig.add_trace(go.Scatter(
        x=reduced[channel_idx, 0],
        y=reduced[channel_idx, 1],
        mode="markers",
        name=f"{vertical} — channel",
        marker=dict(
            symbol="circle",
            size=6,
            color=color,
            opacity=0.55,
            line=dict(width=0.5, color="white")
        ),
        text=[hover_texts[i] for i in channel_idx],
        hoverinfo="text"
    ))

fig.update_layout(
    title=dict(
        text="MMM Vectorstore — t-SNE Visualization<br>"
             "<sup>Stars = summary chunks | Circles = channel chunks | Color = sub-vertical</sup>",
        font=dict(size=16)
    ),
    width=1300,
    height=800,
    plot_bgcolor="#0f172a",
    paper_bgcolor="#0f172a",
    font=dict(color="white"),
    xaxis=dict(title="t-SNE 1", showgrid=False, zeroline=False),
    yaxis=dict(title="t-SNE 2", showgrid=False, zeroline=False),
    legend=dict(
        bgcolor="#1e293b",
        bordercolor="#334155",
        borderwidth=1
    ),
    margin=dict(r=20, b=40, l=40, t=80)
)

fig.show()

['zerbaxia',
 'del-pif',
 'proquad',
 'lynparza',
 'qliftara',
 'vaxneuvance',
 'welireg',
 'vaxelis',
 'januvia',
 'vaqta',
 'rotateq',
 'gardasil',
 'capvaxive',
 'belsomra',
 'keytruda',
 'lenvima',
 'verquovo',
 'dificid',
 'bridion']

In [8]:
import re


In [9]:
merck_brands = []
for meta in collection.get()['metadatas']:
    merck_brands.append(meta['brand'])


merck_channels = set([
    meta['channel'] 
    for meta in collection.get()['metadatas'] 
    if meta['channel'] != "all"
])

list(set(merck_brands))
list(set(merck_channels))

def regex_parser(question: str) -> dict:
    extracted = {}
    question_lower = question.lower()
    brands = set([b for b in merck_brands if re.search(rf'\b{re.escape(b)}\b', question_lower)])
    year   = re.findall(r'\b(202[0-9])\b' , question)
    channel = set([i for i in merck_channels if i in question_lower])
    category = "hcp" if "hcp" in question_lower else ("consumer" if "consumer" in question_lower else None)
    sub_vertical = "oncology" if "oncology" in question_lower else ("vaccines" if "vaccines" in question_lower else ("pharma" if "pharma" in question_lower else None))
    extracted['brands'] = brands
    extracted['year'] = year[0] if year else None
    extracted['channel'] = channel
    extracted['category'] = category
    extracted['sub_vertical'] = sub_vertical
    
    return extracted

In [10]:
print(regex_parser("What is the ROI of Doximity for Proquad in 2024?"))
print(regex_parser("Compare HCP channels for oncology brands"))
print(regex_parser("What is Salesforce calls performance for vaccines in 2023?"))
print(regex_parser("Which consumer channels have the highest ROI?"))

{'brands': {'proquad'}, 'year': '2024', 'channel': {'doximity'}, 'category': None, 'sub_vertical': None}
{'brands': set(), 'year': None, 'channel': set(), 'category': 'hcp', 'sub_vertical': 'oncology'}
{'brands': set(), 'year': '2023', 'channel': set(), 'category': None, 'sub_vertical': 'vaccines'}
{'brands': set(), 'year': None, 'channel': set(), 'category': 'consumer', 'sub_vertical': None}


In [27]:
from datetime import datetime



In [28]:
def llm_parser(question: str) -> dict:
    prompt = f"""
Extract structured filters from this MMM analytics question.
Return ONLY a JSON object with these keys and current year is {datetime.now().year}:':
{{
  "brands": [],          # list of brand names mentioned
  "year": null,          # year as string or null
  "channel": [],         # list of channel/vendor names
  "category": null,      # "hcp" or "consumer" or null
  "sub_vertical": null   # "oncology" or "vaccines" or "pharma" or null
}}

Known brands: {list(merck_brands)}
Known channels: {list(merck_channels)}

Question: {question}

Return only valid JSON. No explanation.
"""
    response = completion(
        model="gemini/gemini-3.1-flash-lite",
        messages=[{"role": "user", "content": prompt}]
    )

    return json.loads(response.choices[0].message.content)


In [29]:
print(llm_parser("What is the ROI of Doximity for Proquad in 2024?"))

{'brands': ['proquad'], 'year': '2024', 'channel': ['doximity'], 'category': None, 'sub_vertical': None}


In [30]:
def parse_query(question: str) -> dict:
    extracted = regex_parser(question)
    
    # check if regex found anything useful
    has_something = any([
        extracted["brands"],
        extracted["year"],
        extracted["channel"],
        extracted["category"],
        extracted["sub_vertical"]
    ])
    
    if not has_something:
        print("Regex found nothing — falling back to LLM parser")
        extracted = llm_parser(question)
    
    return extracted

In [31]:
parse_query("Which brand had the worst model fit last year?")

Regex found nothing — falling back to LLM parser


{'brands': [],
 'year': '2025',
 'channel': [],
 'category': None,
 'sub_vertical': None}

In [ ]:
where = {'brands': ['Capvaxive','Proquad'],
 'year': '2025',
 'channel': [],
 'category': None,
 'sub_vertical': None}
# where={
#     "$or": [
#         {"source": {"$eq": "veeva"}},
#         {"source": {"$eq": "mvcc"}}
#     ]
# }


['Capvaxive', 'Proquad']

In [48]:
def build_where_clause(filters: dict) -> dict:
    conditions = []

    if filters.get("brands"):
        brands_list = list(filters["brands"])
        if len(brands_list) == 1:
            conditions.append({"brand": {"$eq": brands_list[0]}})
        else:
            conditions.append({"brand": {"$in": brands_list}})

    if filters.get("year"):
        conditions.append({"year": {"$eq": filters["year"]}})

    if filters.get("channel"):
        channels_list = list(filters["channel"])
        if len(channels_list) == 1:
            conditions.append({"channel": {"$eq": channels_list[0]}})
        else:
            conditions.append({"channel": {"$in": channels_list}})

    if filters.get("category"):
        conditions.append({"category": {"$eq": filters["category"]}})

    if filters.get("sub_vertical"):
        conditions.append({"sub_vertical": {"$eq": filters["sub_vertical"]}})

    # wrap correctly based on how many conditions
    if len(conditions) == 0:
        return {}
    elif len(conditions) == 1:
        return conditions[0]
    else:
        return {"$and": conditions}

In [49]:
build_where_clause(where)

{'$and': [{'brand': {'$in': ['Capvaxive', 'Proquad']}},
  {'year': {'$eq': '2025'}}]}

In [55]:
def mmm_retriever(question: str, collection, n_results: int = 10) -> str:
    # 1. parse the question
    filters = parse_query(question)

    where = build_where_clause(filters)
    
    # 3. query ChromaDB
    #    if where is empty dict — query without metadata filter
    #    if where has filters — query with where
    if where:
        results = collection.query(
            query_texts=[question],
            n_results=n_results,
            where=where
        )
    else:
        results = collection.query(
            query_texts=[question],
            n_results=n_results
        )
    
    # 4. format results as plain text string
    #    return something the LLM can read
    #    include the chunk text + metadata for each result
    formatted_results = ""
    for i in range(len(results["documents"][0])):
        doc = results["documents"][0][i]
        meta = results["metadatas"][0][i]
        formatted_results += f"--- Result {i+1} ---\n"
        formatted_results += f"Brand: {meta['brand']}\n"
        formatted_results += f"Year: {meta['year']}\n"
        formatted_results += f"Channel: {meta['channel']}\n"
        formatted_results += f"Category: {meta['category']}\n"
        formatted_results += f"Sub-vertical: {meta['sub_vertical']}\n"
        formatted_results += f"Type: {meta['type']}\n"
        formatted_results += f"Content: {doc}\n\n"
    
    return formatted_results

In [56]:
# print(mmm_retriever("What is the ROI of Doximity for Proquad in 2024?", collection))
print(mmm_retriever("Compare HCP channels for oncology brands", collection))

--- Result 1 ---
Brand: lenvima
Year: 2023
Channel: sfmc
Category: hcp
Sub-vertical: oncology
Type: channel
Content: Sfmc is a HCP channel for Lenvima (oncology). Spend was $217,002 with an ROI of 1.83x. Revenue contribution was $397,113. Reach was 14,353 with frequency of 3.9. Adstock decay rate is 0.44. Tactics used: hq_emails.

--- Result 2 ---
Brand: welireg
Year: 2024
Channel: sfmc
Category: hcp
Sub-vertical: oncology
Type: channel
Content: Sfmc is a HCP channel for Welireg (oncology). Spend was $92,932 with an ROI of 2.1x. Revenue contribution was $195,157. Reach was 8,792 with frequency of 4.0. Adstock decay rate is 0.42. Tactics used: hq_emails.

--- Result 3 ---
Brand: lynparza
Year: 2025
Channel: sfmc
Category: hcp
Sub-vertical: oncology
Type: channel
Content: Sfmc is a HCP channel for Lynparza (oncology). Spend was $212,614 with an ROI of 1.95x. Revenue contribution was $414,597. Reach was 21,949 with frequency of 2.2. Adstock decay rate is 0.35. Tactics used: hq_emails.

--

In [59]:
POWER_CURVES = {
    "tv":               0.4,
    "streaming_tv":     0.4,
    "online_video":     0.5,
    "social":           0.6,
    "display":          0.6,
    "audio":            0.6,
    "paid_search":      0.8,   # most linear — intent-based
    "doximity":         0.5,
    "medscape":         0.5,
    "pulsepoint":       0.6,
    "deepintent":       0.6,
    "sfmc":             0.5,
    "nexgen":           0.5,
    "salesforce_calls": 0.3,   # heavy diminishing returns at high frequency
}
input_json = {
    "channel": "tv",
    "current_spend": 3000000,
    "current_revenue_contribution": 4200000,
    "budget_change_pct": -0.20       # -20% cut, +0.10 = 10% increase
}

In [ ]:
def scenario_simulator(input_json: dict) -> dict:
    channel           = input_json["channel"]
    current_spend     = input_json["current_spend"]
    current_revenue   = input_json["current_revenue_contribution"]
    budget_change_pct = input_json["budget_change_pct"]

    power_exponent = POWER_CURVES.get(channel, 0.5)

    new_spend   = current_spend * (1 + budget_change_pct)
    new_revenue = current_revenue * (new_spend / current_spend) ** power_exponent

    # derived metrics
    revenue_change     = new_revenue - current_revenue
    revenue_change_pct = (revenue_change / current_revenue) * 100
    current_roi        = round(current_revenue / current_spend, 2)
    new_roi            = round(new_revenue / new_spend, 2)
    roi_change         = round(new_roi - current_roi, 2)
    spend_change_pct   = budget_change_pct * 100

    # efficiency note — this is what the LLM will read
    direction = "cut" if budget_change_pct < 0 else "increase"
    efficiency_note = (
        f"A {abs(spend_change_pct):.0f}% spend {direction} on {channel} "
        f"leads to a {abs(revenue_change_pct):.1f}% revenue "
        f"{'loss' if revenue_change < 0 else 'gain'} "
        f"(power curve exponent: {power_exponent}). "
        f"ROI moves from {current_roi}x to {new_roi}x. "
    )

    # add diminishing returns insight
    if power_exponent < 0.5 and budget_change_pct < 0:
        efficiency_note += (
            f"Due to strong diminishing returns (exponent {power_exponent}), "
            f"the revenue loss is proportionally smaller than the spend cut — "
            f"efficient channel to reduce."
        )
    elif power_exponent < 0.5 and budget_change_pct > 0:
        efficiency_note += (
            f"Due to strong diminishing returns (exponent {power_exponent}), "
            f"additional spend yields proportionally less revenue — "
            f"consider reallocating to higher-exponent channels."
        )
    elif power_exponent >= 0.7 and budget_change_pct > 0:
        efficiency_note += (
            f"This channel has a relatively linear response (exponent {power_exponent}) "
            f"— additional investment scales efficiently."
        )

    return {
        "channel":                   channel,
        "power_exponent":            power_exponent,
        "current_spend":             round(current_spend, 2),
        "new_spend":                 round(new_spend, 2),
        "spend_change_pct":          round(spend_change_pct, 1),
        "current_revenue":           round(current_revenue, 2),
        "new_revenue":               round(new_revenue, 2),
        "revenue_change":            round(revenue_change, 2),
        "revenue_change_pct":        round(revenue_change_pct, 2),
        "current_roi":               current_roi,
        "new_roi":                   new_roi,
        "roi_change":                roi_change,
        "efficiency_note":           efficiency_note,
    }

In [63]:
scenario_simulator(input_json)

{'channel': 'tv',
 'power_exponent': 0.4,
 'current_spend': 3000000,
 'new_spend': 2400000.0,
 'spend_change_pct': -20.0,
 'current_revenue': 4200000,
 'new_revenue': 3841362.44,
 'revenue_change': -358637.56,
 'revenue_change_pct': -8.54,
 'current_roi': 1.4,
 'new_roi': 1.6,
 'roi_change': 0.2,
 'efficiency_note': 'A 20% spend cut on tv leads to a 8.5% revenue loss (power curve exponent: 0.4). ROI moves from 1.4x to 1.6x. Due to strong diminishing returns (exponent 0.4), the revenue loss is proportionally smaller than the spend cut — efficient channel to reduce.'}

In [66]:
import json
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()

# ─── Benchmark Data ───────────────────────────────────────────────
# Based on aggregated pharma MMM benchmarks (2019-2024)
# Sources: Industry MMM studies across 80+ pharma brands
# ROI = revenue generated per dollar spent

BENCHMARKS = {
    "source": "Aggregated from pharma MMM studies (2019-2024), 80+ brands",
    "last_updated": "2025-01",
    "channels": {

        # ── HCP Digital ───────────────────────────────────────────
        "doximity": {
            "oncology": {"roi_low": 2.2, "roi_mid": 3.0, "roi_high": 3.8, "percentile_75": 3.4},
            "vaccines":  {"roi_low": 1.8, "roi_mid": 2.4, "roi_high": 3.0, "percentile_75": 2.7},
            "pharma":    {"roi_low": 2.0, "roi_mid": 2.7, "roi_high": 3.4, "percentile_75": 3.1},
        },
        "medscape": {
            "oncology": {"roi_low": 2.0, "roi_mid": 2.8, "roi_high": 3.6, "percentile_75": 3.2},
            "vaccines":  {"roi_low": 1.6, "roi_mid": 2.2, "roi_high": 2.8, "percentile_75": 2.5},
            "pharma":    {"roi_low": 1.8, "roi_mid": 2.5, "roi_high": 3.2, "percentile_75": 2.9},
        },
        "pulsepoint": {
            "oncology": {"roi_low": 1.0, "roi_mid": 1.5, "roi_high": 2.0, "percentile_75": 1.8},
            "vaccines":  {"roi_low": 0.9, "roi_mid": 1.3, "roi_high": 1.7, "percentile_75": 1.5},
            "pharma":    {"roi_low": 0.9, "roi_mid": 1.4, "roi_high": 1.8, "percentile_75": 1.6},
        },
        "deepintent": {
            "oncology": {"roi_low": 1.1, "roi_mid": 1.6, "roi_high": 2.1, "percentile_75": 1.9},
            "vaccines":  {"roi_low": 1.0, "roi_mid": 1.4, "roi_high": 1.8, "percentile_75": 1.6},
            "pharma":    {"roi_low": 1.0, "roi_mid": 1.5, "roi_high": 1.9, "percentile_75": 1.7},
        },
        "sfmc": {
            "oncology": {"roi_low": 1.5, "roi_mid": 2.0, "roi_high": 2.6, "percentile_75": 2.3},
            "vaccines":  {"roi_low": 1.3, "roi_mid": 1.8, "roi_high": 2.3, "percentile_75": 2.0},
            "pharma":    {"roi_low": 1.4, "roi_mid": 1.9, "roi_high": 2.4, "percentile_75": 2.1},
        },
        "nexgen": {
            "oncology": {"roi_low": 1.5, "roi_mid": 2.1, "roi_high": 2.7, "percentile_75": 2.4},
            "vaccines":  {"roi_low": 1.4, "roi_mid": 1.9, "roi_high": 2.4, "percentile_75": 2.1},
            "pharma":    {"roi_low": 1.4, "roi_mid": 2.0, "roi_high": 2.5, "percentile_75": 2.2},
        },

        # ── Salesforce ────────────────────────────────────────────
        # Most expensive channel — high absolute revenue but lower ROI
        "salesforce_calls": {
            "oncology": {"roi_low": 1.2, "roi_mid": 1.8, "roi_high": 2.4, "percentile_75": 2.1},
            "vaccines":  {"roi_low": 1.5, "roi_mid": 2.1, "roi_high": 2.7, "percentile_75": 2.4},
            "pharma":    {"roi_low": 1.1, "roi_mid": 1.6, "roi_high": 2.1, "percentile_75": 1.9},
        },

        # ── Consumer ──────────────────────────────────────────────
        "tv": {
            "oncology": {"roi_low": 0.8, "roi_mid": 1.3, "roi_high": 1.8, "percentile_75": 1.6},
            "vaccines":  {"roi_low": 1.8, "roi_mid": 2.5, "roi_high": 3.2, "percentile_75": 2.9},
            "pharma":    {"roi_low": 1.0, "roi_mid": 1.5, "roi_high": 2.0, "percentile_75": 1.8},
        },
        "streaming_tv": {
            "oncology": {"roi_low": 1.0, "roi_mid": 1.5, "roi_high": 2.1, "percentile_75": 1.8},
            "vaccines":  {"roi_low": 2.0, "roi_mid": 2.7, "roi_high": 3.4, "percentile_75": 3.1},
            "pharma":    {"roi_low": 1.2, "roi_mid": 1.7, "roi_high": 2.3, "percentile_75": 2.0},
        },
        "online_video": {
            "oncology": {"roi_low": 1.1, "roi_mid": 1.6, "roi_high": 2.2, "percentile_75": 1.9},
            "vaccines":  {"roi_low": 1.8, "roi_mid": 2.4, "roi_high": 3.0, "percentile_75": 2.7},
            "pharma":    {"roi_low": 1.3, "roi_mid": 1.8, "roi_high": 2.4, "percentile_75": 2.1},
        },
        "paid_search": {
            "oncology": {"roi_low": 2.5, "roi_mid": 3.5, "roi_high": 4.5, "percentile_75": 4.0},
            "vaccines":  {"roi_low": 2.2, "roi_mid": 3.0, "roi_high": 3.8, "percentile_75": 3.4},
            "pharma":    {"roi_low": 2.4, "roi_mid": 3.2, "roi_high": 4.0, "percentile_75": 3.6},
        },
        "social": {
            "oncology": {"roi_low": 1.2, "roi_mid": 1.8, "roi_high": 2.4, "percentile_75": 2.1},
            "vaccines":  {"roi_low": 1.6, "roi_mid": 2.2, "roi_high": 2.8, "percentile_75": 2.5},
            "pharma":    {"roi_low": 1.4, "roi_mid": 2.0, "roi_high": 2.6, "percentile_75": 2.3},
        },
        "display": {
            "oncology": {"roi_low": 0.7, "roi_mid": 1.1, "roi_high": 1.5, "percentile_75": 1.3},
            "vaccines":  {"roi_low": 0.9, "roi_mid": 1.3, "roi_high": 1.7, "percentile_75": 1.5},
            "pharma":    {"roi_low": 0.8, "roi_mid": 1.2, "roi_high": 1.6, "percentile_75": 1.4},
        },
        "audio": {
            "oncology": {"roi_low": 0.8, "roi_mid": 1.2, "roi_high": 1.6, "percentile_75": 1.4},
            "vaccines":  {"roi_low": 1.0, "roi_mid": 1.5, "roi_high": 2.0, "percentile_75": 1.7},
            "pharma":    {"roi_low": 0.9, "roi_mid": 1.3, "roi_high": 1.7, "percentile_75": 1.5},
        },
    },

    # ── Engagement benchmarks (scraped from public sources) ───────
    "engagement": {
        "sfmc": {
            "open_rate": 0.3465,
            "ctr": 0.028,
            "unsubscribe_rate": 0.0025,
            "source": "phamax Digital 2024"
        },
        "nexgen": {
            "open_rate": 0.3465,
            "ctr": 0.028,
            "unsubscribe_rate": 0.0025,
            "source": "phamax Digital 2024"
        },
        "display": {
            "ctr": 0.0035,
            "source": "phamax Digital 2024"
        },
        "paid_search": {
            "ctr": 0.065,
            "source": "phamax Digital 2024"
        },
    }
}

# ─── Scraper — pull live engagement metrics from phamax ───────────
def scrape_engagement_benchmarks() -> dict:
    url = "https://phamax-digital.ch/academy/benchmarks-for-digital-marketing-in-the-pharmaceutical-industry/"
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        # extract all text paragraphs
        paragraphs = [p.get_text(strip=True) for p in soup.find_all("p")]

        # look for benchmark-style lines containing % or rate
        benchmark_lines = [
            p for p in paragraphs
            if any(kw in p.lower() for kw in ["open rate", "click", "ctr", "bounce", "unsubscribe", "%"])
            and len(p) < 200
        ]

        return {
            "source": url,
            "raw_lines": benchmark_lines[:20],   # first 20 relevant lines
            "status": "scraped"
        }

    except Exception as e:
        return {"status": "failed", "error": str(e)}


# ─── Core Benchmark Fetcher ───────────────────────────────────────
def benchmark_fetcher(channel: str, sub_vertical: str, current_roi: float) -> dict:
    channel       = channel.lower().strip()
    sub_vertical  = sub_vertical.lower().strip()

    # look up benchmark
    channel_benchmarks = BENCHMARKS["channels"].get(channel)
    if not channel_benchmarks:
        return {
            "status": "not_found",
            "message": f"No benchmark data available for channel: {channel}",
            "channel": channel,
        }

    vertical_benchmarks = channel_benchmarks.get(sub_vertical)
    if not vertical_benchmarks:
        return {
            "status": "not_found",
            "message": f"No benchmark data for {channel} in {sub_vertical}",
            "channel": channel,
            "sub_vertical": sub_vertical,
        }

    roi_low   = vertical_benchmarks["roi_low"]
    roi_mid   = vertical_benchmarks["roi_mid"]
    roi_high  = vertical_benchmarks["roi_high"]
    p75       = vertical_benchmarks["percentile_75"]

    # derive percentile position of current ROI
    if current_roi >= roi_high:
        percentile_position = "top quartile (>75th percentile)"
        performance_label   = "outperformer"
    elif current_roi >= p75:
        percentile_position = "75th percentile"
        performance_label   = "strong performer"
    elif current_roi >= roi_mid:
        percentile_position = "median to 75th percentile"
        performance_label   = "average performer"
    elif current_roi >= roi_low:
        percentile_position = "below median"
        performance_label   = "underperformer"
    else:
        percentile_position = "bottom quartile (<25th percentile)"
        performance_label   = "significant underperformer"

    # gap to median and p75
    gap_to_median = round(roi_mid - current_roi, 2)
    gap_to_p75    = round(p75 - current_roi, 2)

    # plain English insight for synthesizer
    if current_roi >= roi_mid:
        insight = (
            f"{channel.title()} ROI of {current_roi}x is above the industry median "
            f"of {roi_mid}x for {sub_vertical} brands. "
            f"This channel is performing well — consider protecting or growing this budget."
        )
    else:
        insight = (
            f"{channel.title()} ROI of {current_roi}x is below the industry median "
            f"of {roi_mid}x for {sub_vertical} brands (gap: {abs(gap_to_median):.2f}x). "
            f"Investigate creative quality, targeting, or frequency before increasing spend."
        )

    # pull engagement benchmarks if available
    engagement = BENCHMARKS.get("engagement", {}).get(channel, {})

    return {
        "status":               "found",
        "channel":              channel,
        "sub_vertical":         sub_vertical,
        "current_roi":          current_roi,
        "benchmark_roi_low":    roi_low,
        "benchmark_roi_mid":    roi_mid,
        "benchmark_roi_high":   roi_high,
        "benchmark_p75":        p75,
        "gap_to_median":        gap_to_median,
        "gap_to_p75":           gap_to_p75,
        "percentile_position":  percentile_position,
        "performance_label":    performance_label,
        "insight":              insight,
        "engagement_benchmarks": engagement,
        "data_source":          BENCHMARKS["source"],
    }


# ─── Batch Fetcher — compare all channels in one MMM output ───────
def benchmark_all_channels(channels: list, sub_vertical: str) -> list:
    results = []
    for ch in channels:
        result = benchmark_fetcher(
            channel=ch["name"],
            sub_vertical=sub_vertical,
            current_roi=ch["roi"]
        )
        results.append(result)
    return results


# ─── Entry point ──────────────────────────────────────────────────
if __name__ == "__main__":
    # test single channel
    result = benchmark_fetcher(
        channel="doximity",
        sub_vertical="oncology",
        current_roi=2.4
    )
    print(json.dumps(result, indent=2))

    print("\n" + "="*60)

    # test batch
    sample_channels = [
        {"name": "doximity",         "roi": 2.4},
        {"name": "salesforce_calls", "roi": 1.6},
        {"name": "paid_search",      "roi": 4.1},
        {"name": "tv",               "roi": 1.1},
        {"name": "display",          "roi": 0.9},
    ]
    batch = benchmark_all_channels(sample_channels, "oncology")
    for b in batch:
        if b["status"] == "found":
            print(f"\n{b['channel'].upper()} — {b['performance_label'].title()}")
            print(f"  Current ROI: {b['current_roi']}x  |  Benchmark mid: {b['benchmark_roi_mid']}x")
            print(f"  Position: {b['percentile_position']}")
            print(f"  Insight: {b['insight']}")

    print("\n" + "="*60 + "\nScraping live engagement data...")
    scraped = scrape_engagement_benchmarks()
    print(f"Scrape status: {scraped['status']}")
    if scraped["status"] == "scraped":
        for line in scraped["raw_lines"][:5]:
            print(f"  {line}")

{
  "status": "found",
  "channel": "doximity",
  "sub_vertical": "oncology",
  "current_roi": 2.4,
  "benchmark_roi_low": 2.2,
  "benchmark_roi_mid": 3.0,
  "benchmark_roi_high": 3.8,
  "benchmark_p75": 3.4,
  "gap_to_median": 0.6,
  "gap_to_p75": 1.0,
  "percentile_position": "below median",
  "performance_label": "underperformer",
  "insight": "Doximity ROI of 2.4x is below the industry median of 3.0x for oncology brands (gap: 0.60x). Investigate creative quality, targeting, or frequency before increasing spend.",
  "engagement_benchmarks": {},
  "data_source": "Aggregated from pharma MMM studies (2019-2024), 80+ brands"
}


DOXIMITY — Underperformer
  Current ROI: 2.4x  |  Benchmark mid: 3.0x
  Position: below median
  Insight: Doximity ROI of 2.4x is below the industry median of 3.0x for oncology brands (gap: 0.60x). Investigate creative quality, targeting, or frequency before increasing spend.

SALESFORCE_CALLS — Underperformer
  Current ROI: 1.6x  |  Benchmark mid: 1.8x
  Positi

In [73]:
import sys
print(sys.executable)

/Users/rakshit/auto-analyst/.venv/bin/python


In [75]:
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

In [ ]:
from langfuse.decorators import observe, langfuse_context


In [79]:
import langfuse
print(langfuse.__version__)

4.6.1


In [80]:
import langfuse
print(dir(langfuse))

['BatchEvaluationResult', 'BatchEvaluationResumeToken', 'CompositeEvaluatorFunction', 'Evaluation', 'EvaluatorInputs', 'EvaluatorStats', 'KNOWN_LLM_INSTRUMENTATION_SCOPE_PREFIXES', 'Langfuse', 'LangfuseAgent', 'LangfuseChain', 'LangfuseEmbedding', 'LangfuseEvaluator', 'LangfuseEvent', 'LangfuseGeneration', 'LangfuseGuardrail', 'LangfuseOtelSpanAttributes', 'LangfuseRetriever', 'LangfuseSpan', 'LangfuseTool', 'MapperFunction', 'ObservationTypeLiteral', 'RegressionError', 'RunnerContext', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_client_module', '_task_manager', '_version', 'get_client', 'is_default_export_span', 'is_genai_span', 'is_known_llm_instrumentor', 'is_langfuse_span', 'observe', 'propagate_attributes', 'span_filter']


In [81]:
import langfuse
print(langfuse.__file__)

/Users/rakshit/auto-analyst/.venv/lib/python3.12/site-packages/langfuse/__init__.py


In [85]:
# in your notebook
results = collection.get(where={"brand": {"$eq": "januvia"}})
print(len(results["ids"]), "januvia chunks found")

results = collection.get(where={"$and": [{"brand": {"$eq": "januvia"}}, {"year": {"$eq": "2024"}}]})
print(len(results["ids"]), "januvia 2024 chunks found")

25 januvia chunks found
16 januvia 2024 chunks found


In [88]:
import json

GOLDEN_DATASET = [
    {
        "id": "gtc_001",
        "question": "What is the ROI for Doximity in Januvia's 2024 MMM, and how does it compare to the pharma industry benchmark?",
        "expected_tools": ["mmm_retriever", "benchmark_fetcher"],
        "expected_keywords": ["roi", "doximity", "januvia", "benchmark", "pharma"],
        "ground_truth": "Doximity delivers strong ROI for Januvia in 2024, above the pharma industry midpoint benchmark of 2.7x but below the 75th percentile of 3.1x — a solid above-average performer."
    },
    {
        "id": "gtc_002",
        "question": "Which HCP channel had the highest ROI for Keytruda in 2023?",
        "expected_tools": ["mmm_retriever"],
        "expected_keywords": ["roi", "hcp", "keytruda", "oncology"],
        "ground_truth": "The highest ROI HCP channel for Keytruda in 2023 is identified with its spend and revenue contribution showing it as the most efficient HCP channel."
    },
    {
        "id": "gtc_003",
        "question": "How does Medscape ROI compare across oncology brands, and is it above or below the oncology benchmark?",
        "expected_tools": ["mmm_retriever", "benchmark_fetcher"],
        "expected_keywords": ["roi", "medscape", "oncology", "benchmark"],
        "ground_truth": "Medscape ROI varies across oncology brands with comparison to the oncology benchmark midpoint of 2.8x and 75th percentile of 3.2x."
    },
    {
        "id": "gtc_004",
        "question": "Which channels are generating a negative return on investment (ROI below 1.0) for Dificid in 2024?",
        "expected_tools": ["mmm_retriever", "benchmark_fetcher"],
        "expected_keywords": ["roi", "dificid", "below", "underperforming"],
        "ground_truth": "Channels with ROI below 1.0 for Dificid in 2024 are identified, with recommendations to reallocate budget to higher-performing channels."
    },
    {
        "id": "gtc_005",
        "question": "What is the base versus incremental revenue split for Lenvima in 2023, and what does it imply about media effectiveness?",
        "expected_tools": ["mmm_retriever"],
        "expected_keywords": ["incremental", "base", "lenvima", "media"],
        "ground_truth": "Lenvima 2023 shows a near-even base/incremental split which is unusually media-driven for oncology. Most pharma brands sit at 60-70% base, so high incremental share suggests strong media contribution."
    },
]

In [89]:
def score_tool_accuracy(expected_tools: list, actual_tools: list) -> dict:
    expected_set = set(expected_tools)
    actual_set   = set(actual_tools)

    correct    = expected_set & actual_set        # tools that should have been called and were
    missed     = expected_set - actual_set        # tools that should have been called but weren't
    extra      = actual_set - expected_set        # tools called that weren't needed

    precision  = len(correct) / len(actual_set)  if actual_set  else 0.0
    recall     = len(correct) / len(expected_set) if expected_set else 0.0
    f1         = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return {
        "precision":    round(precision, 2),
        "recall":       round(recall, 2),
        "f1":           round(f1, 2),
        "correct":      list(correct),
        "missed":       list(missed),
        "extra":        list(extra),
    }

# test it
print(score_tool_accuracy(
    expected_tools=["mmm_retriever", "benchmark_fetcher"],
    actual_tools=["mmm_retriever", "benchmark_fetcher", "roi_bar_chart"]
))

{'precision': 0.67, 'recall': 1.0, 'f1': 0.8, 'correct': ['mmm_retriever', 'benchmark_fetcher'], 'missed': [], 'extra': ['roi_bar_chart']}


In [90]:
def score_keyword_coverage(expected_keywords: list, answer_text: str) -> dict:
    answer_lower = answer_text.lower()
    found        = [kw for kw in expected_keywords if kw.lower() in answer_lower]
    missed       = [kw for kw in expected_keywords if kw.lower() not in answer_lower]
    coverage     = round(len(found) / len(expected_keywords), 2) if expected_keywords else 0.0

    return {
        "coverage": coverage,
        "found":    found,
        "missed":   missed,
        "score":    round(coverage * 5, 1),  # scale to 1-5
    }

In [91]:
from litellm import completion

JUDGE_PROMPT = """
You are an expert MMM (Marketing Mix Modeling) analyst evaluating an AI system's answer.

Question: {question}

Ground Truth (what a correct answer looks like):
{ground_truth}

AI System Answer:
{answer}

Score the answer on these three dimensions (1-5 each):

1. FAITHFULNESS — Is the answer grounded in actual data? Does it cite specific numbers or does it hallucinate?
   1 = completely hallucinated, 5 = every claim is data-grounded

2. CORRECTNESS — Does the answer reach the correct analytical conclusion?
   1 = wrong conclusion, 5 = matches ground truth reasoning exactly

3. ACTIONABILITY — Does the answer give a clear recommendation a commercial team can act on?
   1 = vague or no recommendation, 5 = specific, implementable action

Return ONLY valid JSON:
{{
  "faithfulness": <1-5>,
  "correctness": <1-5>,
  "actionability": <1-5>,
  "reasoning": "one sentence explaining the scores"
}}
"""

def score_with_llm_judge(question: str, ground_truth: str, answer: str) -> dict:
    prompt = JUDGE_PROMPT.format(
        question=question,
        ground_truth=ground_truth,
        answer=answer
    )
    response = completion(
        model="gemini/gemini-2.0-flash",
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"```json|```", "", raw).strip()

    try:
        scores = json.loads(raw)
        scores["composite"] = round(
            (scores["faithfulness"] + scores["correctness"] + scores["actionability"]) / 3, 2
        )
        return scores
    except Exception:
        return {
            "faithfulness": 0, "correctness": 0,
            "actionability": 0, "composite": 0,
            "reasoning": "parse error"
        }

In [93]:
SUMMARY_KEYWORDS = [
    "base", "incremental", "r-squared", "mape",
    "total revenue", "model fit", "overall"
]

def needs_summary_chunk(question: str) -> bool:
    q = question.lower()
    return any(kw in q for kw in SUMMARY_KEYWORDS)

In [97]:
from typing import TypedDict, Annotated
from langgraph.graph import add_messages

class AgentState(TypedDict):
    question:      str           # set at start, never changes
    plan:          list          # planner writes this
    tool_results:  list          # executor writes this
    answer:        dict          # synthesizer writes this
    error:         str           # any node can write this

In [102]:
def planner_node(state: AgentState) -> dict:
    question = state["question"]        # read from state
    plan = run_your_planner(question)   # your existing logic
    return {"plan": plan}               # write back to state

In [ ]:
from langgraph.graph import StateGraph, END

# 1. define state
class AgentState(TypedDict):
    question:     str
    plan:         list
    tool_results: list
    answer:       dict

# 2. define nodes (your existing functions, wrapped)
def planner_node(state: AgentState) -> dict:
    question = state["question"]
    plan = run_your_planner(question)
    return {"plan": plan}

def executor_node(state: AgentState) -> dict:
    plan = state["plan"]
    tool_results = run_your_executor(plan)
    return {"tool_results": tool_results}

def synthesizer_node(state: AgentState) -> dict:
    tool_results = state["tool_results"]
    answer = run_your_synthesizer(tool_results)
    return {"answer": answer}

# 3. build the graph
graph = StateGraph(AgentState)

graph.add_node("planner",     planner_node)
graph.add_node("executor",    executor_node)
graph.add_node("synthesizer", synthesizer_node)

# 4. add edges
graph.set_entry_point("planner")          # start here
graph.add_edge("planner", "executor")     # always
graph.add_edge("executor", "synthesizer") # always
graph.add_edge("synthesizer", END)        # done

# 5. compile — this validates the graph and returns a runnable
app = graph.compile()

# 6. run it
result = app.invoke({"question": "What is the ROI of Doximity?"})

In [107]:
class Person(TypedDict):
    name: str
    age: int

In [112]:
a = Person(name="Alice", age=30)

In [113]:
a['age'] = '30'

In [114]:
a

{'name': 'Alice', 'age': '30'}

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END , START

In [121]:
class MyState(TypedDict):
    message: str
    count:str


In [122]:
def node_a(state: MyState) -> dict:
    return {"count": state["count"]+1}


In [123]:
builder = StateGraph(MyState)
builder.add_node("node_a", node_a)
builder.set_entry_point("node_a")
builder.add_edge("node_a", END)

In [124]:
graph = builder.compile()

In [126]:
#4. Run it
result = graph.invoke({"message": "hello", "count": 0})
print(result)  # {"message": "hello", "count": 1}

{'message': 'hello', 'count': 1}


In [ ]:
# Change the state to include a `username` field (str) and a `score` field (int)
# Write a node that appends ` world` to the message field
# Run the graph and verify the output state looks correct


{'message': 'hello', 'count': 1}


In [135]:
class MyState(TypedDict):
    message: str
    count:str
    username: str
    score : int
def node_b(state: MyState) -> dict:
    return {"message": state["message"] + " world"}
builder = StateGraph(MyState)
builder.add_node("node_b", node_b)
builder.set_entry_point("node_b")
builder.add_edge("node_b", END)
graph = builder.compile()
#4. Run it
result = graph.invoke({"message": "hello", "count": 0 ,'username':"rakshit"})
print(result)  # {"message": "hello World", "count": 0}

{'message': 'hello world', 'count': 0, 'username': 'rakshit'}


In [138]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class TextState(TypedDict):
    raw_text: str
    cleaned: str
    result: str
    wc : int

def clean_text(state: TextState):
    text = state["raw_text"].strip().lower()
    return {"cleaned": text}        # only writes "cleaned"

def summarize(state: TextState):
    words = state["cleaned"].split()
    summary = " ".join(words[:5]) + "..."
    return {"result": summary}      # only writes "result"

def word_count(state: TextState):
    return {"wc": len(state["cleaned"].split())}

builder = StateGraph(TextState)
builder.add_node("clean", clean_text)
builder.add_node("word_count", word_count)
builder.add_node("summarize", summarize)

builder.set_entry_point("clean")
builder.add_edge("clean", "word_count")
builder.add_edge("word_count", "summarize")
builder.add_edge("summarize", END) #end

graph = builder.compile()

result = graph.invoke({
    "raw_text": "  Hello World this is LangGraph  ",
    "cleaned": "",
    "result": "",
    "wc": 0
})

print(result)

{'raw_text': '  Hello World this is LangGraph  ', 'cleaned': 'hello world this is langgraph', 'result': 'hello world this is langgraph...', 'wc': 5}


In [146]:
class RefineState(TypedDict):
    topic: str
    draft: str
    attempts: int
    approved: bool
    feedback: str

def generate_draft(state):
    attempt = state["attempts"] + 1
    # in real life: call LLM with state["topic"]
    draft = f"Draft attempt {attempt}: content about {state['topic']} and feedback {state['feedback']}"
    return {"draft": draft, "attempts": attempt}

def evaluate_draft(state):
    # approve after 3 attempts (simulate quality check)
    approved = state["attempts"] >= 3
    feedback = "too short" if len(state["draft"]) < 5 else "looks good"
    return {"approved": approved, "feedback": feedback}

def route_after_eval(state) -> str:
    
    if state["attempts"] >= 5:   # safety: max retries
        return "done"
    if state["approved"]:
        return "done"
    return "retry"

builder = StateGraph(RefineState)
builder.add_node("generate", generate_draft)
builder.add_node("evaluate", evaluate_draft)

builder.set_entry_point("generate")
builder.add_edge("generate", "evaluate")

builder.add_conditional_edges(
    "evaluate",
    route_after_eval,
    {
        "retry": "generate",  # ← THE CYCLE: back to generate
        "done": END
    }
)

graph = builder.compile()
out = graph.invoke({"topic": "AI safety", "draft": "", "attempts": 0, "approved": False, "feedback": ""})
print(out["attempts"], out["draft"])

3 Draft attempt 3: content about AI safety and feedback looks good


In [141]:
# Add a `feedback` field to the state that `evaluate` writes (e.g., 'Too short', 'Good enough')
# Make `generate_draft` read `feedback` and include it in the draft string
# Add a hard cap: if attempts >= 5, always route to END regardless of approval

In [151]:
# drop-in fake LLM for practice — no API key needed
class FakeLLM:
    def bind_tools(self, tools):
        return self
    def invoke(self, messages):
        from langchain_core.messages import AIMessage
        # simulate: first call triggers a tool, second call gives final answer
        if len(messages) < 3:
            return AIMessage(content="", tool_calls=[{
                "name": "get_weather",
                "args": {"city": "Mumbai"},
                "id": "call_001"
            }])
        return AIMessage(content="Mumbai is sunny and 25°C!")

llm = FakeLLM()

In [166]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from typing import TypedDict, Annotated
import operator

# 1. Define a tool
@tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"{city} is sunny and 25°C"

@tool
def get_population(city: str) -> str:
    """Get population for a city."""
    return f"{city} has a population of 20 million"

tools = [get_weather, get_population]
llm = ChatOpenAI(model="gpt-4o-mini").bind_tools(tools)

# 2. State uses Annotated to APPEND messages
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]  # each node appends
    step_count: int

# 3. LLM node
def llm_node(state: AgentState):
    response = llm.invoke(state["messages"])
    return {"messages": [response], "step_count": state["step_count"] + 1}

# 4. Tool execution node
def tools_node(state: AgentState):
    last = state["messages"][-1]
    results = []
    for tc in last.tool_calls:
        result = get_weather.invoke(tc["args"])
        results.append({"role": "tool", "content": result,
                        "tool_call_id": tc["id"]})
    return {"messages": results}

# 5. Router: does the LLM want to call a tool?
def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    return "tools" if getattr(last, "tool_calls", []) else "end" if state["step_count"] > 2 else "end"

builder = StateGraph(AgentState)
builder.add_node("llm", llm_node)
builder.add_node("tools", tools_node)
builder.set_entry_point("llm")
builder.add_conditional_edges("llm", should_continue,
                              {"tools": "tools", "end": END})
builder.add_edge("tools", "llm")  # ← the agent cycle

agent = builder.compile()
out = agent.invoke({"messages": [HumanMessage("whats the weather and population in Delhi?")], "step_count": 0})
print(out["messages"][-1].content)

ImportError: cannot import name 'path_template' from 'openai._utils' (/Users/rakshit/auto-analyst/.venv/lib/python3.12/site-packages/openai/_utils/__init__.py)

In [ ]:
# Add a second tool `get_population(city: str)` that returns a fake population number
# Ask the agent a question that would need both tools: 'What's the weather and population in Delhi?'
# Add a `step_count` field to AgentState and increment it in `llm_node` — verify the agent stops within a reasonable number of steps

In [1]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from typing import TypedDict, Annotated
import operator
from langchain_ollama import ChatOllama

@tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"{city} is sunny and 25°C"

@tool
def get_population(city: str) -> str:
    """Get population for a city."""
    return f"{city} has a population of 20 million"

tools = [get_weather, get_population]
tool_map = {t.name: t for t in tools}          # ← dispatch map
llm = ChatOllama(model="llama3.2:3b").bind_tools(tools)

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    step_count: int

def llm_node(state: AgentState):
    response = llm.invoke(state["messages"])
    return {"messages": [response], "step_count": state["step_count"] + 1}

def tools_node(state: AgentState):
    last = state["messages"][-1]
    print("here")
    print("last message:", last)
    results = []
    for tc in last.tool_calls:
        tool_fn = tool_map[tc["name"]]          # ← correct dispatch
        result = tool_fn.invoke(tc["args"])
        results.append({"role": "tool", "content": result,
                        "tool_call_id": tc["id"]})
    return {"messages": results}

def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if state["step_count"] >= 5:               # safety cap first
        return "end"
    if getattr(last, "tool_calls", []):
        return "tools"
    return "end"

builder = StateGraph(AgentState)
builder.add_node("llm", llm_node)
builder.add_node("tools", tools_node)
builder.set_entry_point("llm")
builder.add_conditional_edges("llm", should_continue,
                              {"tools": "tools", "end": END})
builder.add_edge("tools", "llm")

agent = builder.compile()

out = agent.invoke({
    "messages": [HumanMessage("What's the weather and population in Delhi?")],
    "step_count": 0
})
print("Total messages:", len(out["messages"]))
print("Step count:", out["step_count"])
print()
for i, msg in enumerate(out["messages"]):
    msg_type = type(msg).__name__ if hasattr(msg, '__class__') else 'dict'
    
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        tc_names = [tc['name'] for tc in msg.tool_calls]
        print(f"{i}. [{msg_type}] AIMessage with tool_calls: {tc_names}")
        print(f"   content: {msg.content!r}")
    elif hasattr(msg, 'content'):
        print(f"{i}. [{msg_type}] {msg.content[:100]}")
    elif isinstance(msg, dict):
        print(f"{i}. [tool result] {msg.get('content', '')[:100]}")
    else:
        print(f"{i}. {msg}")

print(out["messages"][-1].content)
print("Total LLM calls:", out["step_count"])

/Users/rakshit/auto-analyst/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


here
last message: content='' additional_kwargs={} response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-05-25T05:48:34.494928Z', 'done': True, 'done_reason': 'stop', 'total_duration': 31297267708, 'load_duration': 25820417750, 'prompt_eval_count': 185, 'prompt_eval_duration': 1491240208, 'eval_count': 36, 'eval_duration': 3921008499, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'} id='lc_run--019e5dad-0eea-75f3-be66-929bc6383c7b-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Delhi'}, 'id': '05e6bb8a-c73a-4ce3-a910-4815e5291dea', 'type': 'tool_call'}, {'name': 'get_population', 'args': {'city': 'Delhi'}, 'id': 'f5c0fab9-e5bb-4d68-9314-428643c459ec', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 185, 'output_tokens': 36, 'total_tokens': 221}
Total messages: 5
Step count: 2

0. [HumanMessage] What's the weather and population in Delhi?
1. [AIMessage] AIMessage with tool_calls: ['get_weather', 'get_population']
  

In [3]:
out['messages']

[HumanMessage(content="What's the weather and population in Delhi?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-05-25T05:48:34.494928Z', 'done': True, 'done_reason': 'stop', 'total_duration': 31297267708, 'load_duration': 25820417750, 'prompt_eval_count': 185, 'prompt_eval_duration': 1491240208, 'eval_count': 36, 'eval_duration': 3921008499, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019e5dad-0eea-75f3-be66-929bc6383c7b-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Delhi'}, 'id': '05e6bb8a-c73a-4ce3-a910-4815e5291dea', 'type': 'tool_call'}, {'name': 'get_population', 'args': {'city': 'Delhi'}, 'id': 'f5c0fab9-e5bb-4d68-9314-428643c459ec', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 185, 'output_tokens': 36, 'total_tokens': 221}),
 {'role': 'tool',
  'content': 'Delhi is sunny and 25°C',
 

In [1]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
import operator

class ChatState(TypedDict):
    messages: Annotated[list[str], operator.add]

def chat_node(state):
    last = state["messages"][-1]
    reply = f"[Bot] You said: {last} (history: {len(state['messages'])} msgs)"
    return {"messages": [reply]}

builder = StateGraph(ChatState)
builder.add_node("chat", chat_node)
builder.set_entry_point("chat")
builder.add_edge("chat", END)

# ← THE NEW PART: pass a checkpointer to compile()
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

# Sessions identified by thread_id
cfg_a = {"configurable": {"thread_id": "session-A"}}

cfg_b = {"configurable": {"thread_id": "session-B"}}
graph.invoke({"messages": ["fresh start"]}, config=cfg_b)

# print("Session A messages:", graph.get_state(cfg_a).values["messages"])
# print("Session B messages:", graph.get_state(cfg_b).values["messages"])

for i in range(1, 6):
    result = graph.invoke({"messages": [f"msg {i}"]}, config=cfg_a)
    snap = graph.get_state(cfg_a)
    print(f"Turn {i}: total messages in state = {len(snap.values['messages'])}")

# Turn 1
r1 = graph.invoke({"messages": ["Hello!"]}, config=cfg_a)
print("Turn 1:", r1["messages"])

r2 = graph.invoke({"messages": ["Hello!"]}, config=cfg_b)
print("Turn 2:", r2["messages"])


# # Turn 2 — same thread_id → state restored
# r2 = graph.invoke({"messages": ["How are you?"]}, config=cfg_a)
# print("Turn 2:", r2["messages"])

# # Inspect saved state
# snapshot = graph.get_state(cfg_a)
# print("\nCurrent state:", snapshot.values)

Turn 1: total messages in state = 2
Turn 2: total messages in state = 4
Turn 3: total messages in state = 6
Turn 4: total messages in state = 8
Turn 5: total messages in state = 10
Turn 1: ['msg 1', '[Bot] You said: msg 1 (history: 1 msgs)', 'msg 2', '[Bot] You said: msg 2 (history: 3 msgs)', 'msg 3', '[Bot] You said: msg 3 (history: 5 msgs)', 'msg 4', '[Bot] You said: msg 4 (history: 7 msgs)', 'msg 5', '[Bot] You said: msg 5 (history: 9 msgs)', 'Hello!', '[Bot] You said: Hello! (history: 11 msgs)']
Turn 2: ['fresh start', '[Bot] You said: fresh start (history: 1 msgs)', 'Hello!', '[Bot] You said: Hello! (history: 3 msgs)']


In [2]:
print("\n=== State history for session-A (newest first) ===")
for i, snap in enumerate(graph.get_state_history(cfg_a)):
    msg_count = len(snap.values.get("messages", []))
    print(f"Checkpoint {i}: {msg_count} messages, next={snap.next}")


=== State history for session-A (newest first) ===
Checkpoint 0: 12 messages, next=()
Checkpoint 1: 11 messages, next=('chat',)
Checkpoint 2: 10 messages, next=('__start__',)
Checkpoint 3: 10 messages, next=()
Checkpoint 4: 9 messages, next=('chat',)
Checkpoint 5: 8 messages, next=('__start__',)
Checkpoint 6: 8 messages, next=()
Checkpoint 7: 7 messages, next=('chat',)
Checkpoint 8: 6 messages, next=('__start__',)
Checkpoint 9: 6 messages, next=()
Checkpoint 10: 5 messages, next=('chat',)
Checkpoint 11: 4 messages, next=('__start__',)
Checkpoint 12: 4 messages, next=()
Checkpoint 13: 3 messages, next=('chat',)
Checkpoint 14: 2 messages, next=('__start__',)
Checkpoint 15: 2 messages, next=()
Checkpoint 16: 1 messages, next=('chat',)
Checkpoint 17: 0 messages, next=('__start__',)


In [62]:
from typing import TypedDict, Optional
from pipeline import TaskPlan,run_planner,build_tool_registry ,TaskPlan ,run_executor,run_synthesizer  # reuse your existing Pydantic model
from langgraph.types import interrupt
from src.db import get_collection


In [63]:
class MMMState(TypedDict):
    question : str
    plan    : Optional[dict]  # planner writes this
    execution_log : Optional[list]     # list of tools called
    answer : Optional[dict]       # synthesizer writes this
    human_feedback : Optional[str]  # human can write this to trigger replan
    error: Optional[str]  # any node can write this
    tools_used: Optional[list]  # track which tools were used




In [6]:
def planner_node(state: MMMState) -> dict:  # MMMstate works like a slate to be written on by various nodes
    question = state["question"]
    plan_object = run_planner(question)  # planner logic returns a Pydantic model (TaskPlan)
    plan_dict = plan_object.model_dump() # convert Pydantic model to dict
    return {'plan': plan_dict} #only updates the 'plan' field in state, other fields remain unchanged



In [ ]:
def human_approval_node(state: MMMState) -> dict:
    # in real life, this would be a UI where a human reviews the plan and either approves or provides feedback
    proposed_plan = state["plan"]    
    feedback = interrupt({
    "question":  state["question"],
    "objective": proposed_plan["objective"],
    "steps":     [f"Step {i+1}: {s['task']}" for i, s in enumerate(proposed_plan["subtasks"])],
    "message":   "Review the plan above. Type 'approve' to proceed or 'reject' to cancel."
                        })
    print(f"> Resumed with input: {feedback}")
    return {"human_feedback": feedback}  # write human feedback back to state  

In [21]:
def route_after_approval(state: MMMState) -> str:
    feedback = state.get("human_feedback", "").lower()
    if feedback == "approve":
        return "executor"  # proceed to execution
    else:
        return "rejected"  
def rejected_node(state: MMMState) -> dict:
    return {"error": f"Plan rejected by human. Feedback: {state.get('human_feedback', 'no feedback')}"}

In [14]:
def executor_node(state: MMMState) -> dict:
    collection = get_collection() 
    tool_registry = build_tool_registry(collection)
    plan = TaskPlan(**state["plan"])
    execution_log = run_executor(plan, tool_registry)
    return {"execution_log": execution_log, "tools_used": [entry["tool_name"] for entry in execution_log]}



    

In [16]:
def synthesizer_node(state: MMMState) -> dict:
    question = state["question"]
    execution_log = state["execution_log"]
    answer = run_synthesizer(question, execution_log)
    return {"answer": answer}
    

In [17]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver

In [25]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()


In [22]:
builder = StateGraph(MMMState)
builder.add_node("planner", planner_node)
builder.add_node("human_approval", human_approval_node)
builder.add_node("rejected", rejected_node)
builder.add_node("executor", executor_node)
builder.add_node("synthesizer", synthesizer_node)
builder.set_entry_point("planner")
builder.add_edge("planner", "human_approval")
builder.add_conditional_edges("human_approval", route_after_approval, {"executor": "executor", "rejected": "rejected"})
builder.add_edge("executor", "synthesizer")
builder.add_edge("synthesizer", END) 
builder.add_edge("rejected", END)



In [ ]:
app = builder.compile(checkpointer=memory)

In [27]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	human_approval(human_approval)
	rejected(rejected)
	executor(executor)
	synthesizer(synthesizer)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	executor --> synthesizer;
	human_approval -.-> executor;
	human_approval -.-> rejected;
	planner --> human_approval;
	rejected --> __end__;
	synthesizer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [28]:
import uuid
thread_id = str(uuid.uuid4())
config    = {"configurable": {"thread_id": thread_id}}

In [29]:
{"question": "What is the ROI of HCP channels for oncology brands?"}

{'question': 'What is the ROI of HCP channels for oncology brands?'}

In [31]:
result = app.invoke({"question": "What is the ROI of HCP channels for oncology brands?"}, config=config)

15:16:18 [INFO] pipeline — Planner: generating task plan
15:16:18 - LiteLLM:INFO: utils.py:4051 - 
LiteLLM completion() model= gemini-3.1-flash-lite; provider = gemini
15:16:18 [INFO] LiteLLM — 
LiteLLM completion() model= gemini-3.1-flash-lite; provider = gemini
15:16:33 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler
15:16:33 [INFO] LiteLLM — Wrapper: Completed Call, calling success_handler
15:16:33 [INFO] pipeline — Planner: objective='Evaluate the ROI of HCP channels for oncology brands and compare them against industry benchmarks.' | subtasks=3
15:16:33 [INFO] pipeline —   step 1: Retrieve MMM data for HCP channels across oncology brands. → mmm_retriever
15:16:33 [INFO] pipeline —   step 2: Fetch industry benchmark ROI for oncology HCP channels. → benchmark_fetcher
15:16:33 [INFO] pipeline —   step 3: Visualize the comparison between current HCP channel ROI and industry benchmarks. → benchmark_comparison_chart


Proposed plan: {'objective': 'Evaluate the ROI of HCP channels for oncology brands and compare them against industry benchmarks.', 'subtasks': [{'task': 'Retrieve MMM data for HCP channels across oncology brands.', 'tool_name': 'mmm_retriever', 'tool_args': {'question': 'What is the ROI of HCP channels for oncology brands?', 'n_results': 15}}, {'task': 'Fetch industry benchmark ROI for oncology HCP channels.', 'tool_name': 'benchmark_fetcher', 'tool_args': {'channels': [{'name': 'HCP'}], 'sub_vertical': 'oncology'}}, {'task': 'Visualize the comparison between current HCP channel ROI and industry benchmarks.', 'tool_name': 'benchmark_comparison_chart', 'tool_args': {'benchmark_results': [], 'sub_vertical': 'oncology'}}], 'reasoning': 'First, we retrieve the MMM data for oncology brands to calculate ROI. Second, we fetch industry benchmarks for HCP channels within the oncology sub-vertical to provide context. Finally, we visualize the ROI comparison using a benchmark comparison chart.'}


In [2]:
from litellm import completion
from src.config import LLM_MODEL


In [3]:
question = "What is the ROI of HCP channels for oncology brands?"

In [ ]:
rewrite_sytem_prompt = """You are a search query optimizer for a pharma MMM vector database.
Rewrite the question as a dense keyword string (15-25 words) 
optimized for semantic similarity search.
Include: brand names, channel names, metrics, time periods, therapeutic area.
Return ONLY the rewritten query string — no explanation. here is the query:"""

def rewrite_query(question: str, attempt_number: int, gap_description: str = "") -> str:

    if attempt_number == 1:
        messages = [
    {"role": "system", "content": rewrite_sytem_prompt},
    {"role": "user",   "content": f"Question: {question}"}]
    
    else:
            retry_prompt = f"""The original question is: {question}
            
                            The previous search was insufficient. The gap identified was: {gap_description}
                            Rewrite the original question to specifically target this missing information."""
            messages = [
                        {"role": "system", "content": rewrite_sytem_prompt},
                        {"role": "user",   "content": retry_prompt}]
    response = completion(
        model = LLM_MODEL,
        messages = messages)
    return response.choices[0].message.content.strip()



In [ ]:
def keyword_check(results: str, filters: dict) -> tuple[bool, str]:
    """
    Check if the results contain the necessary keywords based on the filters.
    Returns a tuple of (is_sufficient, gap_description)
    """
    # filters are brand, year, channel , category , sub_vertical
    passed = []
    missing = []
    passed_checks = 0
    total_checks = 0

    if filters.get("brands"):
        brand_found = any(b.lower() in results.lower() for b in filters["brands"])
        if brand_found:
            passed.append("brand")
            passed_checks += 1
            total_checks += 1
        else:
            missing.append("brand")
            total_checks += 1
    if filters.get("year"):
        year_found = any(str(y) in results for y in filters["year"])
        if year_found:
            passed.append("year")
            passed_checks += 1
            total_checks += 1
        else:
            missing.append("year")
            total_checks += 1
    if filters.get("channels"):
        channel_found = any(c.lower() in results.lower() for c in filters["channels"])
        if channel_found:
            passed.append("channel")
            passed_checks += 1
            total_checks += 1
        else:
            missing.append("channel")
            total_checks += 1
    if filters.get("sub_vertical"):
        sub_vertical_found = any(sv.lower() in results.lower() for sv in filters["sub_vertical"])
        if sub_vertical_found:
            passed.append("sub_vertical")
            passed_checks += 1
            total_checks += 1
        else:
            missing.append("sub_vertical")
            total_checks += 1
    if total_checks == 0:
        return True, "no specific filters to check"

    if passed_checks / total_checks >= 0.5:
        return True, "results look relevant"
    else:
        return False, f"missing: {', '.join(missing)}"
    


In [6]:
import json,re

In [29]:
def llm_relevance_judge(question: str, results: str) -> tuple[bool, str]:
    '''uses an LLM to judge if the results are relevant to the question, and if not, what is missing, return False if insufficient along with a description of the gap'''
    system_prompt = """"You are a data sufficiency auditor for a pharma MMM analytics system.Given a question and retrieved data chunks, decide if the chunks contain 
                 enough specific information to answer the question accurately.You must be strict — if key numbers, brand names, or channel metrics are 
                 missing or belong to the wrong brand/year, return insufficient.
                 Return ONLY valid JSON:
                    {
                    "sufficient": true or false,
                    "gap": "what is missing or empty string if sufficient"
                    }"""
    user_message = f"Question: {question}\n\nRetrieved chunks:\n{results[:2000]}"
    messages = [{"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}]
    response = completion(model = LLM_MODEL, messages=messages)
    raw = response.choices[0].message.content.strip()
    match = re.search(r"\{.*?\}", raw, re.DOTALL) # searching first occurence of {...} in the response to extract the JSON
    if match:
        try:
            result = json.loads(match.group())
            return result.get("sufficient", False), result.get("gap", "no gap description")
        except json.JSONDecodeError:
            return True, ""   
    else:
        return True, ""
        


In [30]:
from difflib import get_close_matches


In [32]:
def did_you_mean(failed_entity:str,known_entities:list) -> str:
    matches = get_close_matches(failed_entity, known_entities, n=1, cutoff=0.6)

    if matches:
        return f"did you mean '{matches[0]}'?"
    else:
        return f"No similar entity found for '{failed_entity}'"
    


In [38]:
from src.query_parser import *

In [39]:
def agentic_retriever(question: str, collection, max_retries: int = 3) -> str:
    #parse the question to extract filters
    known_brands, known_channels = load_known_entities(collection)
    brand_vertical_map = build_brand_vertical_map(collection)
    filters = parse_query(question, known_brands, known_channels, brand_vertical_map)
    attempt = 1
    gap_description = ""
    while attempt <= max_retries:
        print(f"\nAttempt {attempt} with filters: {filters}")
        rewritten_query = rewrite_query(question,attempt,gap_description)
    
         #build where clause from the re-written query and filters
        where = build_where_clause(filters, rewritten_query)

        #query the vector database
        if where:
            results = collection.query(
            query_texts=[rewritten_query],
            n_results=10,
            **({"where": where} if where else {}),
        )
        else:
            results = collection.query(
            query_texts=[rewritten_query],
            n_results=10,
        )
        
        if not results[0]:
            
            failed_entity = re.search(r"brand '(\w+)'", rewritten_query).group(1)
            suggestion = did_you_mean(failed_entity, known_brands)
            return rf"Brand '{failed_entity}' not found. {suggestion}"
    
        parts = []
        for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0]), 1):
            parts.append(
                f"--- Result {i} ---\n"
                f"Brand: {meta['brand']} | Sub-vertical: {meta['sub_vertical']} | "
                f"Year: {meta['year']} | Type: {meta['type']}\n"
                f"Channel: {meta['channel']} | Category: {meta['category']}\n"
                f"Content: {doc}"
            )
        formatted_results = "\n\n".join(parts)

        #run keyword_check
        is_sufficient, gap_description = keyword_check(formatted_results, filters)
        if is_sufficient:
            success,gap_description = llm_relevance_judge(question, formatted_results)
            if success:
                print("LLM judged results sufficient")
                return formatted_results
            else:
                print(f"LLM found results insufficient: {gap_description}")
                attempt += 1    
        else:
            print(f"Keyword check failed: {gap_description}")
            attempt += 1
    return f"Sorry, after {max_retries} attempts, I couldn't retrieve sufficient data to answer your question. Last gap identified: {gap_description}"
            
        
        




        
        